# 04 — Evaluation Pipeline

**Learning objectives:**
- Build test sets from SQuAD
- Run RAGAS metrics
- Interpret evaluation scores

## Setup

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import print_config
print_config()

## Load Test Set

In [ ]:
# ── EXERCISE ──────────────────────────────────────────────────────────────
# Load 5 QA pairs from the SQuAD fallback test set.

# YOUR CODE HERE
testset = ???
print(f"Loaded {???} test samples")
print(testset[0])

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────
from src.evaluation.testset import load_squad_testset

testset = load_squad_testset(sample_size=5)
print(f"Loaded {len(testset)} test samples")
print(testset[0])

## Run Evaluation

In [ ]:
# ── EXERCISE ──────────────────────────────────────────────────────────────
# Run the full RAG pipeline on test questions and evaluate with RAGAS.

# YOUR CODE HERE
# Hint: use build_query_engine, query_with_sources, evaluate_rag

scores = ???
print(scores)

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────
from src.config import FALLBACK_DIR
from src.evaluation.evaluator import evaluate_rag, print_eval_summary
from src.generation.generator import build_query_engine, query_with_sources
from src.ingestion.loader import load_documents
from src.ingestion.chunker import chunk_documents
from src.ingestion.embedder import configure_embed_model
from src.retrieval.store import get_or_create_index

documents = load_documents(FALLBACK_DIR / "squad_sample.json")
embed_model = configure_embed_model()
nodes = chunk_documents(documents, strategy="sentence")
index = get_or_create_index(nodes=nodes, embed_model=embed_model)
engine = build_query_engine(index, nodes=nodes)

questions, answers, contexts, gts = [], [], [], []
for item in testset:
    r = query_with_sources(engine, item["question"])
    questions.append(item["question"])
    answers.append(r["answer"])
    contexts.append([s["text"] for s in r["sources"]] or [item["context"]])
    gts.append(item["ground_truth"])

scores = evaluate_rag(questions, answers, contexts, gts)
print_eval_summary(scores)